<a href="https://colab.research.google.com/github/wan-chtvlv/spatial-coastal-ses/blob/main/B1_1_OceanSpy_tutorial_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **OceanSpy tutorial**

[Use case 2: Particles ](https://oceanspy.readthedocs.io/en/latest/Particles.html#Extract)

---

In this example we will subsample a dataset stored on SciServer using trajectories of synthetic Lagrangian particles. OceanSpy enables the extraction of properties at any given location, but it does not have functionality to compute Lagrangian trajectories yet. The trajectories used in this notebook have been previously computed using a Matlab-based Lagrangian particle code (Gelderloos et al., 2016). The Lagrangian particle code, along with synthetic oceanographic observational platforms (such as Argo floats, isobaric floats, current profiler, and gliders), will be incorporated into OceanSpy in future releases.

The animation below shows the change in temperature of synthetic particles along their 3D trajectories. Particles have been released in Denmark Strait and tracked backward in time from day 0 to day 180, and tracked forward in time from day 180 to 360. All particles are animated forward in time. Source: https://player.vimeo.com/video/296949375

The subsampling method used in this notebook resembles Lagrangian floats deployed by observational oceanographers, such as RAFOS floats. RAFOS floats are isobaric and tracked acoustically. They measure water properties and provide accurate trajectories in both the horizontal and vertical dimensions. The following picture shows the deployment of a RAFOS float during the Overturning in the Subpolar North Atlantic Program (OSNAP). Source: www.o-snap.org

Turns out it needs to use SciServer to run the code on existing notebook. Only works on Jupyter (https://oceanspy.readthedocs.io/en/latest/sciserver.html). I think this will be less helpful for me.

In [1]:
# Display animation
from IPython.display import HTML

# Link
s = "https://player.vimeo.com/video/296949375"

# Options
w = 640
h = 360
f = 0
a = "autoplay; fullscreen"

# Display
HTML(
    f'<iframe src="{s}"'
    'width="640" height="360" frameborder="0" allow="autoplay; fullscreen"'
    "allowfullscreen></iframe>"
)

/usr/local/lib/python3.13/dist-packages/IPython/core/display.py:724: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


#Extract

The computational time to extract particle properties depends on the number of particles, the length of their trajectories, the number of variables requested, and, more importantly, how far the particles travel in the domain and therefore, how dispersed the Lagrangian trajectories are. Thus, it is preferable to extract particle properties asynchronously using the Job mode of SciServer compute (see SciServer access for step-by-step instructions). This notebook, when executed in interactive mode, skips the subsampling step. Instead, it loads an OceanDataset previously created executing the same notebook in Job mode.

`Set interactive = True` to run in Interactive mode, or `interactive = False` to run in Job mode. Alternatively, you can activate the Job mode by typing `interactive = False` in the Parameters slot of the SciServer Job submission form.

In [2]:
interactive = True  # True: Interactive - False: Job

# Check parameters.txt
try:
    # Use values from parameters.txt
    f = open("parameters.txt", "r")
    for line in f:
        exec(line)
    f.close()
except FileNotFoundError:
    # Keep preset values
    pass

In [5]:
# Start Client
from dask.distributed import Client

client = Client()
client

/usr/local/lib/python3.13/dist-packages/distributed/node.py:195: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 43233 instead
  warnings.warn(
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:44295
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:43233/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:39045'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:40519'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:34681 name: 0
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:34681
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:43520
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:38731 name: 1
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:38731
IN

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:43233/status,
Dashboard: http://127.0.0.1:43233/status,Workers: 2
Total threads: 2,Total memory: 12.67 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:44295,Workers: 0
Dashboard: http://127.0.0.1:43233/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:34681,Total threads: 1
Dashboard: http://127.0.0.1:36677/status,Memory: 6.34 GiB
Nanny: tcp://127.0.0.1:39045,


In [6]:
!pip install cartopy
!pip install oceanspy
import cartopy

import matplotlib.pyplot as plt
import numpy as np

# Additional imports
import xarray as xr
from cartopy.crs import PlateCarree

# Import OceanSpy
import oceanspy as ospy

In [10]:
SciServer = True
if SciServer:
  od = ospy.open_oceandataset.from_catalog('get_started')
else:
    import os

    if not os.path.isdir("oceanspy_get_started"):
        # Download get_started
        import subprocess

        print("Downloading and uncompressing get_started data...")
        print("...it might take a couple of minutes.")
        commands = [
            "wget -v -O oceanspy_get_started.tar.gz -L "
            "https://livejohnshopkins-my.sharepoint.com/"
            ":u:/g/personal/malmans2_jh_edu/"
            "EXjiMbANEHBZhy62oUDjzT4BtoJSW2W0tYtS2qO8_SM5mQ?"
            "download=1",
            "tar xvzf oceanspy_get_started.tar.gz",
            "rm -f oceanspy_get_started.tar.gz",
        ]
        subprocess.call("&&".join(commands), shell=True)
    od_snapshot = ospy.open_oceandataset.from_zarr("oceanspy_get_started")

od_snapshot = od_snapshot.compute.potential_density_anomaly()

Opening get_started.


NameError: name 'ValidationError' is not defined

In [7]:


# Open OceanDataset (eulerian fields)
od_eul = ospy.open_oceandataset.from_catalog("EGshelfIIseas2km_ASR_full")

# Mask depth for plotting purposes
Depth = od_eul.dataset["Depth"]
Depth = Depth.where(Depth > 0)
od_eul = od_eul.merge_into_oceandataset(Depth.rename("masked_Depth"))

Opening EGshelfIIseas2km_ASR_full.


NameError: name 'ValidationError' is not defined